# Cohort 3 held-out validation (train Cohorts 1+2, test Cohort 3)

One L1 logistic regression per feature set is fitted on Cohorts 1+2 (C tuned by 5x5
repeated stratified CV on the training cohorts only) and evaluated once on the held-out
Cohort 3. The 95% CI of the test AUC comes from 1,000 bootstrap resamples of the Cohort 3
predictions (the model is not refitted). A label-permutation null (100 permutations of all
labels, LN and LASSO models, 200 bootstrap resamples each) tests whether the held-out AUC
exceeds chance.

**Inputs** (`input_dir`): `counts/counts_norm_cart_paper.tsv.gz`, `metadata/sample_metadata.tsv`
(via `load_modeling_data("all")`), gene sets from `utils/genesets.yaml`.

**Outputs** (`results/ml/holdout_c3/`, used by SFig. 8):
`{LN,FMAC,TEX,LASSO}/{classifier_scores,selected_features,bootstrap_aucs,auc_summary}.csv`,
`perm_{LN,LASSO}/iter_<i>/` (same four files per permutation) and `permutation_aucs.csv`.

**Runtime:** about 40 min with `N_JOBS = 16` (mostly the 100 LASSO permutation fits).

### Train/test split

Train = Cohorts 1+2 (n = 56), test = Cohort 3 (n = 35).
The masks are explicit, and `run_holdout_bootstrap` asserts that they do not overlap.

In [ ]:
import os
import sys

sys.path.insert(0, "../../utils")
import numpy as np
import pandas as pd

from config import res_path
from data_loading import load_modeling_data
from genesets import FMAC_ENSEMBL, LN_ENSEMBL, TEX_ENSEMBL
from modeling import run_holdout_bootstrap, run_holdout_permutations, save_results

In [ ]:
N_BOOTSTRAP = 1000     # bootstrap resamples of the test set, main models
N_PERM = 100           # label permutations
N_BOOTSTRAP_PERM = 200 # bootstrap resamples per permutation fit
N_JOBS = 16

# Output folder inside results_dir (overridable, e.g. CFRNA_ML_DIR=ml_rerun, to keep existing results).
ML_DIR = os.environ.get("CFRNA_ML_DIR", "ml")
OUTDIR = res_path(ML_DIR, "holdout_c3")

MODELS = {                      # name -> run_holdout_bootstrap options
    "LN":    dict(gene_list=LN_ENSEMBL),
    "FMAC":  dict(gene_list=FMAC_ENSEMBL),
    "TEX":   dict(gene_list=TEX_ENSEMBL),
    "LASSO": dict(filter_75=True),
}
PERM_MODELS = {k: MODELS[k] for k in ["LN", "LASSO"]}

## Data and cohort split

In [ ]:
X, y, meta = load_modeling_data("all")
train_mask = meta["cohort"].isin(["Cohort 1", "Cohort 2"])
test_mask = meta["cohort"] == "Cohort 3"
assert not (train_mask & test_mask).any()

print(pd.crosstab(meta["cohort"], y))
print(f"train (C1+C2): {train_mask.sum()} samples | test (C3): {test_mask.sum()} samples")

## Held-out models with bootstrap CI

In [ ]:
holdout = {}
for name, opts in MODELS.items():
    print(f"--- {name}")
    res = run_holdout_bootstrap(X, y, train_mask, test_mask, n_bootstrap=N_BOOTSTRAP, n_jobs=N_JOBS,
                                train_label="C1+C2", test_label="C3", **opts)
    save_results(res, OUTDIR / name)
    holdout[name] = res

summary = pd.DataFrame({name: dict(train_auc=r.train_auc, test_auc=r.test_auc, ci_low=r.ci_low,
                                   ci_high=r.ci_high, best_C=r.best_C, n_selected=len(r.features_df))
                        for name, r in holdout.items()}).T
summary.astype({"n_selected": int}).round(3)

## Label-permutation null

All 91 labels are permuted (one `RandomState(42)` stream), then the
LN and LASSO models are refitted on C1+C2 and scored on C3. The permutation p-value is
(1 + #{permuted AUC >= observed AUC}) / (1 + N_PERM).

In [ ]:
perm_df = run_holdout_permutations(X, y, train_mask, test_mask, PERM_MODELS, n_perm=N_PERM,
                                   n_bootstrap=N_BOOTSTRAP_PERM, outdir=OUTDIR, n_jobs=N_JOBS)
perm_df.to_csv(OUTDIR / "permutation_aucs.csv", index=False)

perm_summary = pd.DataFrame({
    name: dict(observed_test_auc=holdout[name].test_auc,
               perm_median_auc=perm_df[f"{name}_test_auc"].median(),
               perm_p=(1 + (perm_df[f"{name}_test_auc"] >= holdout[name].test_auc).sum()) / (1 + N_PERM))
    for name in PERM_MODELS}).T
perm_summary.round(3)